# Unsupervised learning for patient phenotypes: worked solution

Worked solution for the exercise in the lesson **[Unsupervised learning for patient phenotypes](https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping)** (*Machine learning* track, Medical AI course by Dr. Ram Kumar G). Try the [exercise notebook](09-unsupervised-phenotyping.ipynb) first. Other answers can be just as good: the written parts show one reasonable answer, not the only one.

**In short:** Unsupervised learning looks for structure in data without an outcome label, for example grouping patients into phenotypes. Clustering of six clinical variables has suggested subgroups of adult-onset diabetes in Swedish and Indian populations. But clustering algorithms always return clusters, even from pure noise, so any proposed phenotype must be shown to be stable, reproducible in other populations and more useful than simple clinical measures.

> All data in this notebook are synthetic or openly licensed. Never use real patient data here, and never paste identifiable patient information into public AI tools. Educational use only: the code and models are not medical devices.

## Worked example

These cells reproduce the lesson's code. Lines starting with `#` after the code show the output you should see (numbers can differ slightly with other library versions).

### Clustering synthetic patients

Three synthetic diabetes profiles with overlapping distributions.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(17)
centres = {
    "deficient": dict(age_dx=45, bmi=25, hba1c=10.5, homa_b=40, homa_ir=2.0),
    "resistant": dict(age_dx=50, bmi=34, hba1c=7.5, homa_b=120, homa_ir=5.5),
    "age_related": dict(age_dx=66, bmi=27, hba1c=7.0, homa_b=85, homa_ir=2.5),
}
spread = dict(age_dx=8, bmi=3, hba1c=1.0, homa_b=20, homa_ir=0.9)
frames = [pd.DataFrame({k: rng.normal(v, spread[k], 600) for k, v in c.items()})
          .assign(true_group=name) for name, c in centres.items()]
df = pd.concat(frames, ignore_index=True)
variables = list(spread)
print(df.groupby("true_group")[variables].mean().round(1))

#              age_dx   bmi  hba1c  homa_b  homa_ir
# true_group
# age_related    65.9  26.9    7.0    84.1      2.5
# deficient      44.6  25.2   10.5    38.9      2.0
# resistant      50.4  34.1    7.6   121.2      5.5

### Choosing the number of clusters

Continuing from the previous block: silhouette scores for k = 2 to 6.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df[variables])
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f"k = {k}: silhouette {silhouette_score(X, labels):.2f}")

# k = 2: silhouette 0.39
# k = 3: silhouette 0.45
# k = 4: silhouette 0.34
# k = 5: silhouette 0.25
# k = 6: silhouette 0.16

### Checking against the truth

Continuing from the previous block: compare k = 3 clusters with the true groups.

In [ ]:
from sklearn.metrics import adjusted_rand_score

labels = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
print(pd.crosstab(df.true_group, labels, colnames=["cluster"]))
print(f"\nAgreement with true groups (adjusted Rand index) "
      f"{adjusted_rand_score(df.true_group, labels):.2f}")

# cluster        0    1    2
# true_group
# age_related    4  590    6
# deficient      2    1  597
# resistant    596    4    0
#
# Agreement with true groups (adjusted Rand index) 0.97

### Clustering always finds clusters

k-means on data with no structure, and a stability check.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score

rng = np.random.default_rng(3)
noise = rng.normal(size=(1800, 5))                 # no groups at all
labels = KMeans(n_clusters=4, n_init=10, random_state=0).fit_predict(noise)
print(f"Cluster sizes on pure noise: {np.bincount(labels)}")
print(f"Silhouette {silhouette_score(noise, labels):.2f}")

# Stability: refit on two random halves and compare assignments on the full data
a, b = (KMeans(n_clusters=4, n_init=10, random_state=s)
        .fit(noise[rng.choice(1800, 900, replace=False)]) for s in (1, 2))
print(f"Agreement between two refits (adjusted Rand index) "
      f"{adjusted_rand_score(a.predict(noise), b.predict(noise)):.2f}")

# Cluster sizes on pure noise: [454 446 445 455]
# Silhouette 0.13
# Agreement between two refits (adjusted Rand index) 0.20

## Exercise

**Goal:** Cluster synthetic diabetes profiles, test whether the clusters are real and stable, and compare them with continuous variables for predicting an outcome.

**Data:** Synthetic profiles generated in the notebook (no real patient data) (Generated in the notebook)

### Step 1

Reproduce the silhouette scores and the comparison with the true groups.

In [ ]:
for k in range(2, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=0).fit_predict(X)
    print(f"k = {k}: silhouette {silhouette_score(X, lab):.2f}")
labels3 = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(X)
print(f"k = 3 agreement with the true groups (ARI) {adjusted_rand_score(df.true_group, labels3):.2f}")

### Step 2

Make the groups overlap more (increase the spreads) and watch how silhouette and agreement change.

In [ ]:
def make_profiles(spread_factor, seed=17):
    rng = np.random.default_rng(seed)
    frames = [pd.DataFrame({k: rng.normal(v, spread[k] * spread_factor, 600) for k, v in c.items()})
              .assign(true_group=name) for name, c in centres.items()]
    return pd.concat(frames, ignore_index=True)

for factor in [1.0, 1.5, 2.0, 3.0]:
    d = make_profiles(factor)
    Xd = StandardScaler().fit_transform(d[variables])
    lab = KMeans(n_clusters=3, n_init=10, random_state=0).fit_predict(Xd)
    print(f"spread × {factor}: silhouette {silhouette_score(Xd, lab):.2f}, "
          f"agreement with truth (ARI) {adjusted_rand_score(d.true_group, lab):.2f}")

As the groups overlap, both numbers fall. Real patient data usually look like the more overlapping
settings: a continuum with denser regions, not separate islands. Low silhouette values are a warning that
cluster labels will put many patients in a group they barely belong to.

### Step 3

Run a bootstrap stability analysis: refit k-means on 50 resamples and compute the average adjusted Rand index.

In [ ]:
rng = np.random.default_rng(0)
reference = KMeans(n_clusters=3, n_init=10, random_state=0).fit(X)
scores = []
for b in range(50):
    idx = rng.choice(len(X), len(X), replace=True)
    boot = KMeans(n_clusters=3, n_init=10, random_state=b).fit(X[idx])
    scores.append(adjusted_rand_score(reference.predict(X), boot.predict(X)))
print(f"Bootstrap stability, k = 3: mean ARI {np.mean(scores):.2f} (range {min(scores):.2f}-{max(scores):.2f})")

# The same check for a k that the data do not support
ref5 = KMeans(n_clusters=5, n_init=10, random_state=0).fit(X)
scores5 = [adjusted_rand_score(ref5.predict(X), KMeans(n_clusters=5, n_init=10, random_state=b)
                               .fit(X[rng.choice(len(X), len(X), replace=True)]).predict(X)) for b in range(50)]
print(f"Bootstrap stability, k = 5: mean ARI {np.mean(scores5):.2f}")

With three real, well-separated groups, k = 3 is found again on almost every resample (ARI close to 1).
Forcing five clusters splits real groups in arbitrary places that change from resample to resample, so
stability drops. Stability is a useful check, but not proof: the pure-noise example in the lesson shows
that unstable clusters are a warning sign, while stable ones still need clinical meaning.

### Step 4

Try Gaussian mixture models and hierarchical clustering; do they agree with k-means?

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.mixture import GaussianMixture

methods = {
    "k-means": labels3,
    "Gaussian mixture": GaussianMixture(n_components=3, random_state=0).fit_predict(X),
    "Hierarchical (Ward)": AgglomerativeClustering(n_clusters=3, linkage="ward").fit_predict(X),
}
names = list(methods)
print("Agreement between methods (ARI):")
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        print(f"  {names[i]} vs {names[j]}: {adjusted_rand_score(methods[names[i]], methods[names[j]]):.2f}")
for name, lab in methods.items():
    print(f"{name:<20} vs truth: {adjusted_rand_score(df.true_group, lab):.2f}")

On these well-separated groups the three methods agree closely with each other and with the truth. A good
further experiment: repeat the comparison on the overlapping data from step 2 (spread × 2 or × 3). If the
methods then disagree, the "clusters" reflect each method's assumptions more than the patients.

### Step 5

Simulate a complication outcome that depends on HbA1c and HOMA-IR, and compare a model using cluster labels with one using the continuous variables.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

rng = np.random.default_rng(5)
logit = -4 + 0.6 * (df.hba1c - 8) + 0.5 * (df.homa_ir - 3)
df["complication"] = rng.binomial(1, 1 / (1 + np.exp(-logit)))
df["cluster"] = labels3.astype(str)
print(f"Complication rate {df.complication.mean():.1%}")

train, test = train_test_split(df, test_size=0.3, random_state=0, stratify=df.complication)
X_cl_train, X_cl_test = pd.get_dummies(train.cluster), pd.get_dummies(test.cluster)
cluster_model = LogisticRegression().fit(X_cl_train, train.complication)
continuous_model = LogisticRegression(max_iter=1000).fit(train[variables], train.complication)
print(f"Cluster labels only:  AUROC {roc_auc_score(test.complication, cluster_model.predict_proba(X_cl_test)[:, 1]):.3f}")
print(f"Continuous variables: AUROC {roc_auc_score(test.complication, continuous_model.predict_proba(test[variables])[:, 1]):.3f}")

Cluster labels throw away the variation inside each cluster: two patients in the same cluster with HbA1c
8% and 12% get the same prediction. For predicting an individual's outcome, the continuous measurements
usually do better; clusters can still be useful for describing a population or designing trials.

---

Lesson: [https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping](https://ramkumarmd.com/medical-ai/machine-learning/unsupervised-phenotyping). Code in this notebook is released under the MIT licence; the text under CC BY 4.0. See the repository README for details.